# ⚔️ Sistema de RPG — Arquitetura e Persistência
**Disciplina:** Desenvolvimento de Sistemas  |  **Ferramentas:** Python (Colab), SQLite, UML

**Equipe:** _(preencha os nomes)_

> ▶️ **Como executar:** menu *Ambiente de execução → Executar tudo*. A célula 7 recria o banco `rpg.db` do zero, então pode rodar quantas vezes quiser.

### Arquitetura em camadas
| Camada | Responsabilidade | Células |
|---|---|---|
| Domínio | Entidades, regras de negócio, exceções | 1 a 5 |
| Persistência | SQLite (DDL + CRUD) | 6 |
| Aplicação | Orquestração do jogo / PoC | 7 a 9 |
| Qualidade | Testes automatizados das regras e do banco | 10 |

### Os 4 pilares de POO neste projeto
- **Abstração:** `Personagem`, `Item` e `IUsavel` são classes abstratas (`ABC`).
- **Encapsulamento:** vida, mana, XP, ouro e capacidade do inventário são privados; o estado só muda por métodos que validam as regras.
- **Herança:** `Guerreiro`, `Mago`, `Arqueiro` e `Monstro` herdam de `Personagem`; `PocaoVida`, `PocaoMana`, `ElixirXP` herdam de `Item`.
- **Polimorfismo:** `atacar()`, `habilidade_especial()` e `usar()` têm comportamentos diferentes em cada subclasse.

## 1. Exceções de domínio

In [1]:
class RPGError(Exception):
    """Exceção base do domínio do RPG."""

class InventarioCheioError(RPGError): pass
class OuroInsuficienteError(RPGError): pass
class ItemNaoEncontradoError(RPGError): pass
class ManaInsuficienteError(RPGError): pass
class PersonagemMortoError(RPGError): pass
class ValorInvalidoError(RPGError): pass        # quantidades negativas, zero ou não inteiras
class PersonagemJaExisteError(RPGError): pass   # nome já usado por outra classe de personagem
class CombateInvalidoError(RPGError): pass      # ex.: lutar contra monstro já derrotado


def _positivo(valor, campo="valor"):
    """Valida que 'valor' é um inteiro estritamente positivo."""
    if isinstance(valor, bool) or not isinstance(valor, int) or valor <= 0:
        raise ValorInvalidoError(f"{campo} deve ser um inteiro positivo (recebido: {valor!r}).")
    return valor

## 2. Itens: interface `IUsavel` e hierarquia de `Item`

In [2]:
from abc import ABC, abstractmethod
import random
import sqlite3


class IUsavel(ABC):
    """Interface: tudo que pode ser usado por um personagem."""
    @abstractmethod
    def usar(self, alvo: "Personagem") -> str: ...


class Item(IUsavel, ABC):
    TIPO = "Item"

    def __init__(self, nome: str, preco: int, valor_efeito: int):
        self._nome = nome
        self._preco = preco
        self._valor_efeito = valor_efeito

    @property
    def nome(self): return self._nome
    @property
    def preco(self): return self._preco
    @property
    def valor_efeito(self): return self._valor_efeito
    @property
    def tipo(self): return self.TIPO

    def __repr__(self):
        return f"{self.tipo}({self._nome}, {self._preco} ouro)"


class PocaoVida(Item):
    TIPO = "PocaoVida"
    def usar(self, alvo):
        curado = alvo.curar(self._valor_efeito)
        return f"{alvo.nome} bebeu {self.nome} e recuperou {curado} de vida."


class PocaoMana(Item):
    TIPO = "PocaoMana"
    def usar(self, alvo):
        rec = alvo.recuperar_mana(self._valor_efeito)
        return f"{alvo.nome} bebeu {self.nome} e recuperou {rec} de mana."


class ElixirXP(Item):
    TIPO = "ElixirXP"
    def usar(self, alvo):
        subiu = alvo.ganhar_xp(self._valor_efeito)
        return (f"{alvo.nome} usou {self.nome} e ganhou {self._valor_efeito} XP."
                + (" ⬆️ SUBIU DE NÍVEL!" if subiu else ""))


# Fábrica: converte o "tipo" salvo no banco na classe correta (polimorfismo + persistência)
TIPOS_ITEM = {c.TIPO: c for c in (PocaoVida, PocaoMana, ElixirXP)}

CATALOGO = [
    PocaoVida("Poção de Vida Pequena", 20, 30),
    PocaoVida("Poção de Vida Grande", 50, 80),
    PocaoMana("Poção de Mana", 30, 40),
    ElixirXP("Elixir de Experiência", 60, 50),
]

## 3. Inventário (gerenciamento de coleções)

In [3]:
class Inventario:
    """Coleção de itens com capacidade limitada (conta unidades)."""

    def __init__(self, capacidade: int = 10):
        self._capacidade = _positivo(capacidade, "capacidade")
        self._itens: dict[str, list] = {}   # nome -> [Item, quantidade]

    @property
    def capacidade(self): return self._capacidade

    @property
    def total_unidades(self):
        return sum(qtd for _, qtd in self._itens.values())

    def adicionar(self, item: Item, quantidade: int = 1):
        _positivo(quantidade, "quantidade")
        if self.total_unidades + quantidade > self._capacidade:
            raise InventarioCheioError(
                f"Inventário cheio ({self.total_unidades}/{self._capacidade}).")
        if item.nome in self._itens:
            self._itens[item.nome][1] += quantidade
        else:
            self._itens[item.nome] = [item, quantidade]

    def obter(self, nome: str) -> Item:
        """Consulta o item SEM removê-lo."""
        if nome not in self._itens:
            raise ItemNaoEncontradoError(f"'{nome}' não está no inventário.")
        return self._itens[nome][0]

    def remover(self, nome: str) -> Item:
        item = self.obter(nome)
        if self._itens[nome][1] == 1:
            del self._itens[nome]
        else:
            self._itens[nome][1] -= 1
        return item

    def listar(self):
        return [(item, qtd) for item, qtd in self._itens.values()]

    def __len__(self): return self.total_unidades

## 4. Personagens: classe abstrata, herança e polimorfismo

In [4]:
class Personagem(ABC):
    CUSTO_ESPECIAL = 0
    JOGAVEL = True          # False para NPCs (não são salvos no banco)

    def __init__(self, nome, vida_max, mana_max, forca, ouro=100, capacidade=10):
        self._nome = nome
        self._vida_max = vida_max
        self._vida = vida_max
        self._mana_max = mana_max
        self._mana = mana_max
        self._forca = forca
        self._xp = 0
        self._nivel = 1
        self._ouro = ouro
        self._inventario = Inventario(capacidade)

    # ---------- encapsulamento: leitura via properties ----------
    @property
    def nome(self): return self._nome
    @property
    def vida(self): return self._vida
    @property
    def vida_max(self): return self._vida_max
    @property
    def mana(self): return self._mana
    @property
    def mana_max(self): return self._mana_max
    @property
    def forca(self): return self._forca
    @property
    def xp(self): return self._xp
    @property
    def nivel(self): return self._nivel
    @property
    def ouro(self): return self._ouro
    @property
    def vivo(self): return self._vida > 0
    @property
    def classe(self): return type(self).__name__
    @property
    def inventario(self): return self._inventario      # somente leitura (sem setter)

    # ---------- regras de negócio (única forma de alterar o estado) ----------
    def receber_dano(self, dano: int) -> int:
        if isinstance(dano, bool) or not isinstance(dano, int) or dano < 0:
            raise ValorInvalidoError(f"dano inválido: {dano!r}")
        self._vida = max(0, self._vida - dano)
        return dano

    def curar(self, qtd: int) -> int:
        _positivo(qtd, "cura")
        if not self.vivo:
            raise PersonagemMortoError(f"{self.nome} está morto.")
        antes = self._vida
        self._vida = min(self._vida_max, self._vida + qtd)
        return self._vida - antes

    def gastar_mana(self, qtd: int):
        _positivo(qtd, "mana")
        if self._mana < qtd:
            raise ManaInsuficienteError(f"{self.nome} não tem mana ({self._mana}/{qtd}).")
        self._mana -= qtd

    def recuperar_mana(self, qtd: int) -> int:
        _positivo(qtd, "mana")
        antes = self._mana
        self._mana = min(self._mana_max, self._mana + qtd)
        return self._mana - antes

    def ganhar_ouro(self, qtd: int):
        self._ouro += _positivo(qtd, "ouro")

    def gastar_ouro(self, qtd: int):
        _positivo(qtd, "ouro")
        if qtd > self._ouro:
            raise OuroInsuficienteError(
                f"Ouro insuficiente: precisa de {qtd}, possui {self._ouro}.")
        self._ouro -= qtd

    def ganhar_xp(self, qtd: int) -> bool:
        """Soma XP e sobe de nível a cada (nivel*100) XP. Retorna True se subiu.
        Ao subir, os máximos aumentam e o personagem ganha o bônus (sem cura total)."""
        self._xp += _positivo(qtd, "xp")
        subiu = False
        while self._xp >= self._nivel * 100:
            self._xp -= self._nivel * 100
            self._nivel += 1
            self._vida_max += 10
            self._mana_max += 5
            self._forca += 2
            self._vida += 10
            self._mana += 5
            subiu = True
        return subiu

    def usar_item(self, nome_item: str) -> str:
        item = self.inventario.obter(nome_item)   # só consulta
        msg = item.usar(self)                     # polimorfismo; se falhar, o item NÃO é perdido
        self.inventario.remover(nome_item)        # consome apenas após o efeito dar certo
        return msg

    # ---------- métodos abstratos: cada classe implementa do seu jeito ----------
    @abstractmethod
    def atacar(self, alvo: "Personagem") -> int: ...

    @abstractmethod
    def habilidade_especial(self, alvo: "Personagem") -> int: ...

    def _restaurar_estado(self, nivel, vida_max, vida, mana_max, mana, forca, xp, ouro):
        """Usado SOMENTE pela camada de persistência para reidratar o objeto."""
        self._nivel, self._vida_max, self._vida = nivel, vida_max, vida
        self._mana_max, self._mana, self._forca = mana_max, mana, forca
        self._xp, self._ouro = xp, ouro

    def __str__(self):
        return (f"{self.classe} {self.nome} | Nv {self.nivel} | HP {self.vida}/{self.vida_max} "
                f"| MP {self.mana}/{self.mana_max} | XP {self.xp} | Ouro {self.ouro}")


class Guerreiro(Personagem):
    CUSTO_ESPECIAL = 10
    def __init__(self, nome, capacidade=10):
        super().__init__(nome, vida_max=120, mana_max=30, forca=15, capacidade=capacidade)
    def atacar(self, alvo):
        return alvo.receber_dano(self._forca + random.randint(0, 5))
    def habilidade_especial(self, alvo):          # Golpe Poderoso
        self.gastar_mana(self.CUSTO_ESPECIAL)
        return alvo.receber_dano(self._forca * 2)


class Mago(Personagem):
    CUSTO_ESPECIAL = 20
    def __init__(self, nome, capacidade=10):
        super().__init__(nome, vida_max=70, mana_max=100, forca=6, capacidade=capacidade)
    def atacar(self, alvo):
        return alvo.receber_dano(self._forca + random.randint(0, 3))
    def habilidade_especial(self, alvo):          # Bola de Fogo
        self.gastar_mana(self.CUSTO_ESPECIAL)
        return alvo.receber_dano(self._forca * 2 + 25)


class Arqueiro(Personagem):
    CUSTO_ESPECIAL = 12
    def __init__(self, nome, capacidade=10):
        super().__init__(nome, vida_max=90, mana_max=50, forca=11, capacidade=capacidade)
    def atacar(self, alvo):
        return alvo.receber_dano(self._forca + random.randint(2, 8))
    def habilidade_especial(self, alvo):          # Chuva de Flechas (3 disparos)
        self.gastar_mana(self.CUSTO_ESPECIAL)
        return sum(alvo.receber_dano(self._forca // 2 + random.randint(0, 4)) for _ in range(3))


class Monstro(Personagem):
    JOGAVEL = False
    def __init__(self, nome, vida, forca, recompensa_xp=40, recompensa_ouro=25):
        super().__init__(nome, vida_max=vida, mana_max=0, forca=forca, ouro=0)
        self.recompensa_xp = recompensa_xp
        self.recompensa_ouro = recompensa_ouro
    def atacar(self, alvo):
        return alvo.receber_dano(self._forca + random.randint(0, 4))
    def habilidade_especial(self, alvo):
        return self.atacar(alvo)


CLASSES_PERSONAGEM = {c.__name__: c for c in (Guerreiro, Mago, Arqueiro)}

## 5. Combate e Loja (regras de negócio)

In [5]:
class Combate:
    """Ciclo completo de combate por turnos."""

    def __init__(self, heroi: Personagem, monstro: Monstro):
        self.heroi, self.monstro = heroi, monstro
        self.log: list[str] = []

    def executar(self) -> bool:
        h, m = self.heroi, self.monstro
        if not h.vivo:
            raise PersonagemMortoError(f"{h.nome} está morto e não pode lutar.")
        if not m.vivo:
            raise CombateInvalidoError(f"{m.nome} já foi derrotado.")
        self.log.append(f"⚔️  {h.nome} ({h.classe}) vs {m.nome}")
        turno = 0
        while h.vivo and m.vivo:
            turno += 1
            # herói: usa habilidade especial quando tem mana, senão ataque básico
            try:
                dano = h.habilidade_especial(m)
                acao = "habilidade especial"
            except ManaInsuficienteError:
                dano = h.atacar(m)
                acao = "ataque básico"
            self.log.append(f"  T{turno}: {h.nome} usa {acao} → {dano} dano (monstro HP {m.vida})")
            if not m.vivo:
                break
            dano_m = m.atacar(h)
            self.log.append(f"  T{turno}: {m.nome} ataca → {dano_m} dano (herói HP {h.vida})")

        if h.vivo:
            subiu = h.ganhar_xp(m.recompensa_xp)
            h.ganhar_ouro(m.recompensa_ouro)
            self.log.append(f"🏆 Vitória! +{m.recompensa_xp} XP, +{m.recompensa_ouro} ouro."
                            + (" ⬆️ SUBIU DE NÍVEL!" if subiu else ""))
        else:
            self.log.append(f"💀 {h.nome} foi derrotado.")
        return h.vivo


class Loja:
    def __init__(self, catalogo):
        self._catalogo = {i.nome: i for i in catalogo}

    def listar(self):
        return list(self._catalogo.values())

    def comprar(self, comprador: Personagem, nome_item: str) -> Item:
        if nome_item not in self._catalogo:
            raise ItemNaoEncontradoError(f"'{nome_item}' não existe na loja.")
        item = self._catalogo[nome_item]
        # ordem importa: valida inventário ANTES de cobrar (evita perder ouro sem receber o item)
        if comprador.inventario.total_unidades >= comprador.inventario.capacidade:
            raise InventarioCheioError("Inventário cheio, compra cancelada.")
        comprador.gastar_ouro(item.preco)           # pode lançar OuroInsuficienteError
        comprador.inventario.adicionar(item)
        return item

## 6. Persistência — SQLite (DDL + CRUD)

**Modelo relacional (3FN):**
- `personagens` (1) ── (N) `inventario` (N) ── (1) `itens`
- `inventario` é a tabela associativa (resolve o N:N entre personagens e itens) com FK para ambas.
- Apenas personagens **jogáveis** são salvos (`Monstro` é NPC).

In [6]:
class RepositorioRPG:
    def __init__(self, caminho="rpg.db"):
        self.conn = sqlite3.connect(caminho)
        self.conn.execute("PRAGMA foreign_keys = ON")   # SQLite não ativa FK por padrão!
        self._criar_tabelas()

    def fechar(self):
        self.conn.close()

    def _criar_tabelas(self):
        self.conn.executescript("""
        CREATE TABLE IF NOT EXISTS personagens (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            nome       TEXT    NOT NULL UNIQUE,
            classe     TEXT    NOT NULL,
            nivel      INTEGER NOT NULL,
            vida_max   INTEGER NOT NULL,
            vida       INTEGER NOT NULL,
            mana_max   INTEGER NOT NULL,
            mana       INTEGER NOT NULL,
            forca      INTEGER NOT NULL,
            xp         INTEGER NOT NULL,
            ouro       INTEGER NOT NULL,
            capacidade INTEGER NOT NULL
        );
        CREATE TABLE IF NOT EXISTS itens (
            id           INTEGER PRIMARY KEY AUTOINCREMENT,
            nome         TEXT    NOT NULL UNIQUE,
            tipo         TEXT    NOT NULL,
            preco        INTEGER NOT NULL CHECK (preco >= 0),
            valor_efeito INTEGER NOT NULL
        );
        CREATE TABLE IF NOT EXISTS inventario (
            personagem_id INTEGER NOT NULL,
            item_id       INTEGER NOT NULL,
            quantidade    INTEGER NOT NULL CHECK (quantidade > 0),
            PRIMARY KEY (personagem_id, item_id),
            FOREIGN KEY (personagem_id) REFERENCES personagens(id) ON DELETE CASCADE,
            FOREIGN KEY (item_id)       REFERENCES itens(id)
        );
        """)
        self.conn.commit()

    # ---------- itens ----------
    def semear_itens(self, catalogo):
        self.conn.executemany(
            "INSERT OR IGNORE INTO itens (nome, tipo, preco, valor_efeito) VALUES (?,?,?,?)",
            [(i.nome, i.tipo, i.preco, i.valor_efeito) for i in catalogo])
        self.conn.commit()

    # ---------- CREATE / UPDATE ----------
    def salvar(self, p: Personagem):
        if not p.JOGAVEL:
            raise RPGError(f"{p.classe} não é um personagem jogável e não pode ser salvo.")
        with self.conn:   # transação atômica: tudo ou nada
            row = self.conn.execute(
                "SELECT id, classe FROM personagens WHERE nome=?", (p.nome,)).fetchone()
            if row and row[1] != p.classe:
                raise PersonagemJaExisteError(
                    f"O nome '{p.nome}' já pertence a um {row[1]}; escolha outro nome.")
            dados = (p.nivel, p.vida_max, p.vida, p.mana_max, p.mana, p.forca,
                     p.xp, p.ouro, p.inventario.capacidade)
            if row:                                   # UPDATE explícito (não consome ids)
                pid = row[0]
                self.conn.execute("""UPDATE personagens SET nivel=?, vida_max=?, vida=?,
                    mana_max=?, mana=?, forca=?, xp=?, ouro=?, capacidade=? WHERE id=?""",
                    dados + (pid,))
            else:                                     # INSERT
                pid = self.conn.execute("""INSERT INTO personagens
                    (nome, classe, nivel, vida_max, vida, mana_max, mana, forca, xp, ouro, capacidade)
                    VALUES (?,?,?,?,?,?,?,?,?,?,?)""", (p.nome, p.classe) + dados).lastrowid
            # inventário: regrava (e cadastra automaticamente itens ainda desconhecidos)
            self.conn.execute("DELETE FROM inventario WHERE personagem_id=?", (pid,))
            for item, qtd in p.inventario.listar():
                self.conn.execute(
                    "INSERT OR IGNORE INTO itens (nome, tipo, preco, valor_efeito) VALUES (?,?,?,?)",
                    (item.nome, item.tipo, item.preco, item.valor_efeito))
                iid = self.conn.execute("SELECT id FROM itens WHERE nome=?", (item.nome,)).fetchone()[0]
                self.conn.execute(
                    "INSERT INTO inventario (personagem_id, item_id, quantidade) VALUES (?,?,?)",
                    (pid, iid, qtd))

    # ---------- READ ----------
    def carregar(self, nome: str) -> Personagem:
        row = self.conn.execute("""
            SELECT id, classe, capacidade, nivel, vida_max, vida, mana_max, mana, forca, xp, ouro
            FROM personagens WHERE nome=?""", (nome,)).fetchone()
        if row is None:
            raise ItemNaoEncontradoError(f"Personagem '{nome}' não encontrado no banco.")
        pid, classe, capacidade, *estado = row
        p = CLASSES_PERSONAGEM[classe](nome, capacidade)   # fábrica: recria a subclasse certa
        p._restaurar_estado(*estado)
        for tipo, inome, preco, efeito, qtd in self.conn.execute("""
            SELECT i.tipo, i.nome, i.preco, i.valor_efeito, inv.quantidade
            FROM inventario inv JOIN itens i ON i.id = inv.item_id
            WHERE inv.personagem_id=?""", (pid,)):
            p.inventario.adicionar(TIPOS_ITEM[tipo](inome, preco, efeito), qtd)
        return p

    def listar_personagens(self):
        return self.conn.execute(
            "SELECT id, nome, classe, nivel, vida, ouro FROM personagens").fetchall()

    # ---------- DELETE ----------
    def excluir(self, nome: str):
        with self.conn:
            self.conn.execute("DELETE FROM personagens WHERE nome=?", (nome,))  # CASCADE limpa inventário

## 7. PoC — Parte 1: criar o banco, o herói e comprar itens
> Esta célula apaga e recria o `rpg.db`, então a demonstração sempre começa do zero.

In [7]:
import os
import pandas as pd
from IPython.display import display

# fecha conexões de execuções anteriores (se existirem) antes de recriar o arquivo
for _c in ("repo", "repo2"):
    try: globals()[_c].conn.close()
    except Exception: pass

random.seed(42)                     # resultados reproduzíveis na apresentação
if os.path.exists("rpg.db"): os.remove("rpg.db")

repo = RepositorioRPG("rpg.db")
repo.semear_itens(CATALOGO)
loja = Loja(CATALOGO)

heroi = Guerreiro("Arthas")
print(heroi)

print("\n🛒 Compras:")
for nome in ["Poção de Vida Grande", "Poção de Mana", "Poção de Vida Pequena"]:
    item = loja.comprar(heroi, nome)
    print(f"  comprou {item.nome} → ouro restante: {heroi.ouro}")

# Tratamento de exceção 1: saldo insuficiente
try:
    loja.comprar(heroi, "Elixir de Experiência")
except OuroInsuficienteError as e:
    print("\n❌ Erro tratado:", e)

repo.salvar(heroi)
print("\n💾 Estado salvo no SQLite.")

Guerreiro Arthas | Nv 1 | HP 120/120 | MP 30/30 | XP 0 | Ouro 100

🛒 Compras:
  comprou Poção de Vida Grande → ouro restante: 50
  comprou Poção de Mana → ouro restante: 20
  comprou Poção de Vida Pequena → ouro restante: 0

❌ Erro tratado: Ouro insuficiente: precisa de 60, possui 0.

💾 Estado salvo no SQLite.


## 8. PoC — Parte 2: combate, derrota, uso de item e nova persistência

In [8]:
orc = Monstro("Orc Selvagem", vida=90, forca=12, recompensa_xp=120, recompensa_ouro=40)
combate = Combate(heroi, orc)
combate.executar()
print("\n".join(combate.log))

print("\n", heroi)
print(heroi.usar_item("Poção de Vida Grande"))

# Tratamento de exceção 2: item inexistente
try:
    heroi.usar_item("Espada Lendária")
except ItemNaoEncontradoError as e:
    print("❌ Erro tratado:", e)

# Tratamento de exceção 3: inventário cheio (capacidade definida no construtor)
pequeno = Mago("Merlin", capacidade=1)
pequeno.ganhar_ouro(500)
loja.comprar(pequeno, "Poção de Mana")
try:
    loja.comprar(pequeno, "Poção de Mana")
except InventarioCheioError as e:
    print("❌ Erro tratado:", e)

# Tratamento de exceção 4: lutar contra monstro já derrotado
try:
    Combate(heroi, orc).executar()
except CombateInvalidoError as e:
    print("❌ Erro tratado:", e)

# Cenário de DERROTA: mago iniciante contra um dragão
aprendiz = Mago("Aprendiz")
dragao = Monstro("Dragão Ancião", vida=400, forca=30, recompensa_xp=500, recompensa_ouro=300)
c2 = Combate(aprendiz, dragao)
c2.executar()
print("\n" + "\n".join(c2.log[:1] + c2.log[-2:]))

# Item não é perdido se o efeito falhar (herói morto não pode ser curado)
aprendiz.inventario.adicionar(CATALOGO[0])
try:
    aprendiz.usar_item("Poção de Vida Pequena")
except PersonagemMortoError as e:
    print("❌ Erro tratado:", e, "| itens preservados:", len(aprendiz.inventario))

repo.salvar(heroi)
repo.salvar(Arqueiro("Legolas"))
print("\n💾 Estado salvo novamente.")

⚔️  Arthas (Guerreiro) vs Orc Selvagem
  T1: Arthas usa habilidade especial → 30 dano (monstro HP 60)
  T1: Orc Selvagem ataca → 12 dano (herói HP 108)
  T2: Arthas usa habilidade especial → 30 dano (monstro HP 30)
  T2: Orc Selvagem ataca → 12 dano (herói HP 96)
  T3: Arthas usa habilidade especial → 30 dano (monstro HP 0)
🏆 Vitória! +120 XP, +40 ouro. ⬆️ SUBIU DE NÍVEL!

 Guerreiro Arthas | Nv 2 | HP 106/130 | MP 5/35 | XP 20 | Ouro 40
Arthas bebeu Poção de Vida Grande e recuperou 24 de vida.
❌ Erro tratado: 'Espada Lendária' não está no inventário.
❌ Erro tratado: Inventário cheio, compra cancelada.
❌ Erro tratado: Orc Selvagem já foi derrotado.

⚔️  Aprendiz (Mago) vs Dragão Ancião
  T3: Dragão Ancião ataca → 31 dano (herói HP 0)
💀 Aprendiz foi derrotado.
❌ Erro tratado: Aprendiz está morto. | itens preservados: 1

💾 Estado salvo novamente.


## 9. PoC — Parte 3: comprovação visual da persistência (SELECT nas tabelas)

In [9]:
print("TABELA personagens")
display(pd.read_sql("SELECT * FROM personagens", repo.conn))

print("TABELA itens")
display(pd.read_sql("SELECT * FROM itens", repo.conn))

print("TABELA inventario (com JOIN legível)")
display(pd.read_sql("""
    SELECT p.nome AS personagem, i.nome AS item, inv.quantidade
    FROM inventario inv
    JOIN personagens p ON p.id = inv.personagem_id
    JOIN itens i       ON i.id = inv.item_id
""", repo.conn))

# CARREGAR: abre uma conexão NOVA, como se o jogo tivesse sido fechado e reaberto
repo2 = RepositorioRPG("rpg.db")
recuperado = repo2.carregar("Arthas")
print("\n📂 Personagem carregado do banco:")
print(recuperado)
print("Inventário:", recuperado.inventario.listar())
print("É Guerreiro?", isinstance(recuperado, Guerreiro))

# DELETE
repo2.excluir("Legolas")
print("\nApós excluir Legolas:", repo2.listar_personagens())
repo2.fechar()

TABELA personagens


,id,nome,classe,nivel,vida_max,vida,mana_max,mana,forca,xp,ouro,capacidade
0,1,Arthas,Guerreiro,2,130,130,35,5,17,20,40,10
1,2,Legolas,Arqueiro,1,90,90,50,50,11,0,100,10


TABELA itens


,id,nome,tipo,preco,valor_efeito
0,1,Poção de Vida Pequena,PocaoVida,20,30
1,2,Poção de Vida Grande,PocaoVida,50,80
2,3,Poção de Mana,PocaoMana,30,40
3,4,Elixir de Experiência,ElixirXP,60,50


TABELA inventario (com JOIN legível)


,personagem,item,quantidade
0,Arthas,Poção de Mana,1
1,Arthas,Poção de Vida Pequena,1



📂 Personagem carregado do banco:
Guerreiro Arthas | Nv 2 | HP 130/130 | MP 5/35 | XP 20 | Ouro 40
Inventário: [(PocaoVida(Poção de Vida Pequena, 20 ouro), 1), (PocaoMana(Poção de Mana, 30 ouro), 1)]
É Guerreiro? True

Após excluir Legolas: [(1, 'Arthas', 'Guerreiro', 2, 130, 40)]


## 10. Testes automatizados
Cada teste abaixo protege uma regra de negócio ou um cenário de borda. Usa banco em memória (`:memory:`), portanto não interfere no `rpg.db` da demonstração.

In [10]:
def deve_falhar(excecao, fn):
    try:
        fn()
    except excecao:
        return
    raise AssertionError(f"esperava {excecao.__name__}")

testes = []
def teste(f):
    testes.append(f); return f

@teste
def poção_nao_e_perdida_se_efeito_falha():
    h = Guerreiro("T1"); h.inventario.adicionar(CATALOGO[0]); h.receber_dano(999)
    deve_falhar(PersonagemMortoError, lambda: h.usar_item("Poção de Vida Pequena"))
    assert len(h.inventario) == 1

@teste
def valores_invalidos_sao_rejeitados():
    h = Guerreiro("T2")
    for fn in (lambda: h.gastar_ouro(-500), lambda: h.ganhar_ouro(0), lambda: h.ganhar_xp(-1),
               lambda: h.curar(-5), lambda: h.receber_dano(-1),
               lambda: h.inventario.adicionar(CATALOGO[0], -5)):
        deve_falhar(ValorInvalidoError, fn)
    assert h.ouro == 100 and len(h.inventario) == 0

@teste
def monstro_nao_e_salvo():
    deve_falhar(RPGError, lambda: RepositorioRPG(":memory:").salvar(Monstro("Orc", 90, 12)))

@teste
def capacidade_do_inventario_persiste():
    r = RepositorioRPG(":memory:"); r.salvar(Mago("M", capacidade=3))
    assert r.carregar("M").inventario.capacidade == 3

@teste
def item_desconhecido_e_cadastrado_ao_salvar():
    r = RepositorioRPG(":memory:")                  # sem semear_itens
    g = Guerreiro("G"); g.inventario.adicionar(PocaoVida("Poção X", 1, 1)); r.salvar(g)
    assert r.carregar("G").inventario.listar()[0][0].nome == "Poção X"

@teste
def nome_duplicado_de_outra_classe_e_rejeitado():
    r = RepositorioRPG(":memory:"); r.salvar(Guerreiro("Igual"))
    deve_falhar(PersonagemJaExisteError, lambda: r.salvar(Mago("Igual")))
    assert r.listar_personagens()[0][2] == "Guerreiro" and r.carregar("Igual").vida_max == 120

@teste
def ids_sao_sequenciais_apos_varios_saves():
    r = RepositorioRPG(":memory:"); a = Guerreiro("A")
    for _ in range(3): r.salvar(a)
    r.salvar(Mago("B"))
    assert [x[0] for x in r.listar_personagens()] == [1, 2]

@teste
def monstro_morto_nao_luta_de_novo():
    h, m = Guerreiro("H"), Monstro("M", 10, 1)
    Combate(h, m).executar(); xp = h.xp
    deve_falhar(CombateInvalidoError, lambda: Combate(h, m).executar())
    assert h.xp == xp

@teste
def loja_nao_cobra_se_inventario_cheio():
    p = Mago("P", capacidade=1); loja_t = Loja(CATALOGO)
    loja_t.comprar(p, "Poção de Mana"); ouro = p.ouro
    deve_falhar(InventarioCheioError, lambda: loja_t.comprar(p, "Poção de Mana"))
    assert p.ouro == ouro

@teste
def salvar_e_carregar_preserva_estado_e_tipo():
    r = RepositorioRPG(":memory:"); a = Arqueiro("Arq"); a.inventario.adicionar(CATALOGO[2], 2)
    a.ganhar_xp(250); a.ganhar_ouro(7); r.salvar(a); b = r.carregar("Arq")
    assert isinstance(b, Arqueiro)
    assert (b.nivel, b.xp, b.ouro, b.vida_max) == (a.nivel, a.xp, a.ouro, a.vida_max)
    assert len(b.inventario) == 2

@teste
def excluir_remove_inventario_em_cascata():
    r = RepositorioRPG(":memory:"); g = Guerreiro("Del"); g.inventario.adicionar(CATALOGO[0])
    r.salvar(g); r.excluir("Del")
    assert r.conn.execute("SELECT COUNT(*) FROM inventario").fetchone()[0] == 0

for t in testes:
    t()
    print(f"✅ {t.__name__}")
print(f"\n🎉 {len(testes)} testes passaram.")

✅ poção_nao_e_perdida_se_efeito_falha
✅ valores_invalidos_sao_rejeitados
✅ monstro_nao_e_salvo
✅ capacidade_do_inventario_persiste
✅ item_desconhecido_e_cadastrado_ao_salvar
✅ nome_duplicado_de_outra_classe_e_rejeitado
✅ ids_sao_sequenciais_apos_varios_saves
✅ monstro_morto_nao_luta_de_novo
✅ loja_nao_cobra_se_inventario_cheio
✅ salvar_e_carregar_preserva_estado_e_tipo
✅ excluir_remove_inventario_em_cascata

🎉 11 testes passaram.
